# LlamaIndex + Llama.cpp

## Configuration

In [ ]:
#https://docs.llamaindex.ai/en/stable/understanding/rag/
#https://docs.llamaindex.ai/en/stable/examples/low_level/oss_ingestion_retrieval/

In [ ]:
!pip install llama-index-readers-file pymupdf

In [ ]:
!pip install llama-index-vector-stores-chroma

In [ ]:
!pip install llama-index-embeddings-huggingface

In [ ]:
!pip install llama-index-llms-llama-cpp

In [ ]:
# !pip install opencv-python-headless==4.8.0.74

In [ ]:
!pip install llama-cpp-python

In [ ]:
!pip show llama-index

## Model

**llama-2-chat-13b-ggml model**

In [ ]:
# Importing the LlamaCPP class from the llama_index library
from llama_index.llms.llama_cpp import LlamaCPP

# URL to the pre-trained Llama 2 13B model hosted on Hugging Face
model_url = "https://huggingface.co/TheBloke/Llama-2-13B-chat-GGUF/resolve/main/llama-2-13b-chat.Q4_0.gguf"

# Initializing the LlamaCPP model with specified parameters
llm = LlamaCPP(
    model_url=model_url,  # Set the URL to download the model
    model_path=None,  # If a local model path is provided, it will be used instead of downloading
    temperature=0,  # Controls randomness in generation (0 = deterministic output)
    max_new_tokens=256,  # Limits the maximum number of new tokens generated in a response
    context_window=4096,  # Defines the maximum number of tokens the model can consider at once (Llama 2 default)
    generate_kwargs={},  # Additional keyword arguments for the model's generation function
    model_kwargs={"n_gpu_layers": 1},  # Allocates at least 1 layer to GPU to enable acceleration
    verbose=True,  # Enables detailed logging for debugging and monitoring
)


**all-MiniLM-L12-v2** embedding model

In [ ]:
# sentence transformers
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L12-v2") #https://www.sbert.net/docs/sentence_transformer/pretrained_models.html

#### Chromadb configuration

**EphemeralClient** is suitable for in-memory storage and does not persist data to disk.

To save data to disk using DuckDB/Parquet, we need to use **PersistentClient** or configure the persist_directory in the settings properly.

In [ ]:
# from llama_index.vector_stores.chroma import ChromaVectorStore
# from chromadb.config import Settings
# import chromadb

# # create client and a new collection
# chroma_client = chromadb.EphemeralClient()
# chroma_collection = chroma_client.create_collection("Collection1")


# # Initialize the Chroma vector store
# vector_store = ChromaVectorStore(
#     persist_directory="./chroma_data",  # Directory to save ChromaDB data
#     chroma_collection=chroma_collection,
#     embedding_dim=384  # Dimension of OpenAI embeddings
# )

**OR**

After trying the code using the first method (EphemeralClient), it didn't work!
Data needs to be saved in disk to allow access then

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from chromadb import Client
from chromadb.config import Settings
import chromadb

#before  https://docs.trychroma.com/production/administration/migration
# Configure ChromaDB for persistent storage
# chroma_settings = Settings(
#     chroma_db_impl="duckdb+parquet",
#     persist_directory="./chroma_data"  # Directory to save ChromaDB data
# )
# Create a Persistent Client
# chroma_client = Client(settings=chroma_settings)

#after
chroma_client = chromadb.PersistentClient(path="./chroma_data")

# Create or get a collection
chroma_collection = chroma_client.get_or_create_collection("Collection")

# Initialize the Chroma vector store
vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection,
    persist_directory="./chroma_data",  # Directory to save ChromaDB data
    embedding_dim=384
)


## Data Indexing

### Loading and splitting data

In [ ]:
# Importing Path from pathlib to handle file paths
from pathlib import Path

# Importing PyMuPDFReader from llama_index to read PDF files
from llama_index.readers.file import PyMuPDFReader

# Importing SentenceSplitter from llama_index to split text into smaller chunks
from llama_index.core.node_parser import SentenceSplitter


# Define a list of file paths to process
file_paths = [
    "./OPEN SOURCE SOFTWARE GUIDELINES.pdf",  # Path to the first PDF file
    "./Creating Impactful.pdf"  # Path to the second PDF file
]

# Initialize the PDF loader
loader = PyMuPDFReader()

# Initialize the text parser with chunking parameters
text_parser = SentenceSplitter(
    chunk_size=1000,  # Maximum number of characters per chunk
    chunk_overlap=200,  # Overlap between chunks to maintain context
    separator=" "  # Separator to split text (default: space)
)

# Initialize lists to store text chunks and document indices
text_chunks = []  # Stores extracted text chunks
doc_idxs = []  # Stores the corresponding document index for each chunk

# Iterate through the list of file paths and process each document
for file_idx, file_path in enumerate(file_paths):
    try:
        # Load the document from the given file path
        documents = loader.load(file_path=file_path)

        # Iterate through each document (some PDFs may contain multiple text objects)
        for doc_idx, doc in enumerate(documents):
            # Split the document text into smaller chunks
            cur_text_chunks = text_parser.split_text(doc.text)

            # Extend the main text_chunks list with the extracted chunks
            text_chunks.extend(cur_text_chunks)

            # Store the file index for each chunk to track its source
            doc_idxs.extend([file_idx] * len(cur_text_chunks))

    # Handle cases where the file is not found
    except FileNotFoundError:
        print(f"File not found: {file_path}")

    # Catch any other unexpected errors and display an error message
    except Exception as e:
        print(f"An error occurred while loading {file_path}: {e}")

# Debugging: Print the total number of chunks extracted
print(f"Total number of chunks: {len(text_chunks)}")

# Debugging: Print the first chunk from each file if chunks exist
if text_chunks:
    print(f"First chunk from {file_paths[0]}: {text_chunks[2]}")  # First chunk from the first document


In [ ]:
text_chunks

### Manually Construct Nodes from Text Chunks

In [ ]:
# from llama_index.core.schema import TextNode

# nodes = []
# for idx, doc in enumerate(text_chunks):
#     # Add metadata if it exists
#     if hasattr(doc, 'metadata'):
#         print(doc.metadata)


In [ ]:
from llama_index.core.schema import TextNode

nodes = []
for idx, doc in enumerate(text_chunks):
    # Extract text content from the Document instance
    text_content = doc.page_content if hasattr(doc, 'page_content') else str(doc)
    node = TextNode(
        text=text_content,  # Ensure this is a string
    )
    nodes.append(node)


###  Generate Embeddings for each Node

In [ ]:
# Iterate over each node in the list of nodes
for node in nodes:
    # Generate an embedding for the node's content using the embedding model
    node_embedding = embed_model.get_text_embedding(
        node.get_content()  # Extract text content
    )
    # Assign the generated embedding to the node's 'embedding' attribute
    node.embedding = node_embedding


In [ ]:
# Check the total number of nodes
batch_size = len(nodes)
print(f"Batch size (number of nodes): {batch_size}")

### Load Nodes into a Vector Store

the batch size of nodes (413) exceeds the maximum batch size allowed by ChromaDB (166). ChromaDB imposes a limit on the number of items you can add to a collection in a single batch. To resolve this issue, you can split your data into smaller batches and add them iteratively.

**Avoid running this multiple times because it adds the nodes each time to the vector_store!!**

In [ ]:
for idx, node in enumerate(nodes):
    try:
        vector_store.add([node])
    except Exception as e:
        print(f"Error with node {idx}: {e}")


**The number of nodes in the collection must much the number of nodes existed**

In [ ]:
# Get the count of nodes in the collection
node_count = chroma_collection.count()
print(f"Number of nodes: {node_count}")

In [ ]:
for item in chroma_collection.get():
    print(item)

In [ ]:
# Inspecting contents of the vector store
try:
    # Retrieve all stored items
    all_items = vector_store.client.get()
    # print("Contents of the vector store:")
    # for item in all_items["documents"]:
    #     print(item)
except Exception as e:
    print(f"Error retrieving vector store contents: {e}")

### Querying

In [ ]:
# Define the query string for which we want to retrieve relevant information
query_str = "Can you tell me about what is OSS"

# Generate an embedding (vector representation) for the query using the embedding model
query_embedding = embed_model.get_query_embedding(query_str)


In [ ]:
# Import the VectorStoreQuery class from llama_index to perform vector-based similarity search
from llama_index.core.vector_stores import VectorStoreQuery

# Define the query mode (default mode typically performs a standard similarity search)
query_mode = "default"

# Construct a query object for the vector store
vector_store_query = VectorStoreQuery(
    query_embedding=query_embedding,  # Use the generated embedding of the query string
    similarity_top_k=2,  # Retrieve the top 2 most similar results based on vector similarity
    mode=query_mode  # Specify the search mode (e.g., default, exact match, hybrid, etc.)
)

In [ ]:
# Execute the query on the vector store to retrieve relevant nodes
query_result = vector_store.query(vector_store_query)

# Print the content of the most relevant (top-ranked) retrieved node
# print(query_result.nodes[0].get_content()) 

In [ ]:
# Import NodeWithScore class to store nodes along with their similarity scores
from llama_index.core.schema import NodeWithScore
from typing import Optional  # Import Optional for type hinting (score can be None)

# Initialize an empty list to store nodes with their similarity scores
nodes_with_scores = []

# Iterate over the retrieved nodes from the query result
for index, node in enumerate(query_result.nodes):
    # Initialize score as None (in case similarities are not available)
    score: Optional[float] = None

    # If similarity scores are available, assign the corresponding score to the node
    if query_result.similarities is not None:
        score = query_result.similarities[index]

    # Create a NodeWithScore object and add it to the list
    nodes_with_scores.append(NodeWithScore(node=node, score=score))

* **\__init\__ method:**

**vector_store:** The vector store where data is stored and from which the retriever will fetch relevant results.

**embed_model:** The embedding model used to generate the vector embeddings for queries.

**query_mode:** Defines how the query should be processed (e.g., similarity-based search).

**similarity_top_k:** Determines how many top results should be returned based on the similarity.

* **\_retrieve method:**

**query_embedding:** The query string is embedded using the embedding model to convert it into a numerical vector representation.

**vector_store_query:** Constructs a query object that contains the query embedding, the number of top results to retrieve, and the query mode.

**query_result:** Executes the query against the vector store to retrieve relevant nodes.

**nodes_with_scores:** Loops through the result and pairs each node with its corresponding similarity score, storing them in a list.

**NodeWithScore:** Each node is paired with its similarity score, making it easier to rank and process later.

In [ ]:
# Import necessary classes and types
from llama_index.core import QueryBundle
from llama_index.core.retrievers import BaseRetriever
from typing import Any, List

# Define the VectorDBRetriever class which extends BaseRetriever
class VectorDBRetriever(BaseRetriever):
    """Retriever over a postgres vector store."""

    # Initialize the retriever with necessary parameters
    def __init__(
        self,
        vector_store: ChromaVectorStore,  # The vector store to retrieve from (e.g., a Chroma vector store)
        embed_model: embed_model,  # The embedding model used for generating query embeddings
        query_mode: str = "default",  # Mode of query, default is "default"
        similarity_top_k: int = 2,  # Number of top results to return based on similarity
    ) -> None:
        """Initialization of the retriever parameters."""
        self._vector_store = vector_store  # Store the vector store instance
        self._embed_model = embed_model  # Store the embedding model instance
        self._query_mode = query_mode  # Store the query mode (e.g., "default"): query string ->  query embedding compared with stored embeddings -> retrun similar vectors
        self._similarity_top_k = similarity_top_k  # Store the number of top results to retrieve
        super().__init__()  # Call the initializer of the base class (BaseRetriever)

    # Define the retrieve method to query the vector store and get relevant nodes
    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]: #query_bundle: all the necessary information related to the query
        """Retrieve relevant nodes based on the query."""

        # Generate the embedding for the query string from the embedding model
        query_embedding = self._embed_model.get_query_embedding(
            query_bundle.query_str  # Extract the query string from the query bundle
        )

        # Create a query for the vector store using the query embedding
        vector_store_query = VectorStoreQuery(
            query_embedding=query_embedding,  # Use the query embedding for search
            similarity_top_k=self._similarity_top_k,  # Limit to the top k most similar results
            mode=self._query_mode,  # Define the query mode (default search behavior)
        )

        # Execute the query on the vector store to retrieve results
        query_result = self._vector_store.query(vector_store_query)

        # List to store the nodes and their similarity scores
        nodes_with_scores = []

        # Iterate through the query result's nodes
        for index, node in enumerate(query_result.nodes):
            # Initialize the score as None in case similarities are not available
            score: Optional[float] = None
            if query_result.similarities is not None:
                score = query_result.similarities[index]  # Retrieve the similarity score for the node

            # Append the node along with its score to the nodes_with_scores list
            nodes_with_scores.append(NodeWithScore(node=node, score=score))

        # Return the list of nodes along with their similarity scores
        return nodes_with_scores


In [ ]:
# Initialize the VectorDBRetriever object
retriever = VectorDBRetriever(
    vector_store,  # The vector store that holds the document embeddings to search against
    embed_model,   # The embedding model that will generate vector representations of the query text
    query_mode="default",  # The mode for handling the query, here it is set to "default", implying a standard similarity search
    similarity_top_k=2     # The number of top results to retrieve based on similarity (in this case, 2 most similar results)
)

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

# Initialize the RetrieverQueryEngine using the retriever and the LLM (Large Language Model)
query_engine = RetrieverQueryEngine.from_args(
    retriever,  # The retriever that will be used to search the vector store for relevant nodes
    llm=llm     # The large language model (LLM) used for generating or processing responses based on the retrieved nodes
)

In [ ]:
# query_str1 = "What is the definition of Open Source Software?"

# response1 = query_engine.query(query_str1)

In [ ]:
# print(str(response1))

In [ ]:
# print(response1.source_nodes[0].get_content())

In [ ]:
# query_str2 = "What is early determination of distribution policy?"

# response2 = query_engine.query(query_str2)


In [ ]:
# print(response2.source_nodes[0].get_content())

# Evaluation

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes, embed_model= embed_model)


### Dataset Generation


In [ ]:
from llama_index.core.schema import BaseNode
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core import ChatPromptTemplate, PromptTemplate
from typing import Tuple, List
import re


In [ ]:
# Import necessary classes from LlamaIndex
from llama_index.core.schema import BaseNode  # Base class for indexing nodes
from llama_index.core.llms import ChatMessage, MessageRole  # Handles chat messages and roles
from llama_index.core import ChatPromptTemplate, PromptTemplate  # Used for formatting prompts
from typing import Tuple, List  # Import type hints for function parameters
import re  # Import regex module (not used in the current code)

# Define a prompt template for question answering
QA_PROMPT = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"  # Placeholder for context information
    "---------------------\n"
    "Given the context information and not prior knowledge, "  
    "answer the query.\n"
    "Query: {query_str}\n"  # Placeholder for the question
    "Answer: "  # The model is expected to generate the answer after this
)

def generate_answers_for_questions(
    questions: List[str], context: str, llm: llm
) -> List[str]:  # Function returns a list of answers
    """Generate answers for questions given a specific context using the LLM."""

    answers = []  # List to store generated answers

    for question in questions:  # Loop through each question
        # Format the prompt by injecting the context and question into the template
        fmt_qa_prompt = QA_PROMPT.format(
            context_str=context, query_str=question
        )

        # Use the LLM to generate an answer for the given prompt
        response_obj = llm.complete(fmt_qa_prompt)

        # Convert the response object to a string and store it in the answers list
        answers.append(str(response_obj))

    return answers  # Return the list of generated answers


The function generate_qa_pairs takes nodes (which contain content from documents), an LLM, and a parameter num_questions_per_chunk (default: 10).

For each node (document chunk):

* The content of the node is extracted using node.get_content(metadata_mode="all").

* The system and user prompts are formatted using question_gen_template.format_messages:

    * System Message (QUESTION_GEN_SYS_TMPL): Instructs the LLM to act as a professor and generate diverse questions from the given content.

    * User Message (QUESTION_GEN_USER_TMPL): Provides the actual document chunk as context and asks the LLM to generate relevant questions.

* The LLM is called using llm.chat(fmt_messages), and it returns a set of generated questions.

* These questions are cleaned using regex (re.sub(r"^\d+[\).\s]", "", question).strip()) to remove numbering.

In [ ]:
QUESTION_GEN_USER_TMPL = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "generate the relevant questions. "
)

QUESTION_GEN_SYS_TMPL = """\
You are a Teacher/ Professor. Your task is to setup \
{num_questions_per_chunk} questions for an upcoming \
quiz/examination. The questions should be diverse in nature \
across the document. Restrict the questions to the \
context information provided.\
"""

question_gen_template = ChatPromptTemplate(
    message_templates=[
        ChatMessage(role=MessageRole.SYSTEM, content=QUESTION_GEN_SYS_TMPL),
        ChatMessage(role=MessageRole.USER, content=QUESTION_GEN_USER_TMPL),
    ]
)


def generate_qa_pairs(
    nodes: List[BaseNode], llm: llm, num_questions_per_chunk: int = 10
) -> List[Tuple[str, str]]:
    """Generate questions."""
    qa_pairs = []
    for idx, node in enumerate(nodes):
        print(f"Node {idx}/{len(nodes)}")
        context_str = node.get_content(metadata_mode="all")
        fmt_messages = question_gen_template.format_messages(
            num_questions_per_chunk=10,
            context_str=context_str,
        )
        chat_response = llm.chat(fmt_messages)
        raw_output = chat_response.message.content
        result_list = str(raw_output).strip().split("\n")
        cleaned_questions = [
            re.sub(r"^\d+[\).\s]", "", question).strip()
            for question in result_list
        ]
        answers = generate_answers_for_questions(
            cleaned_questions, context_str, llm
        )
        cur_qa_pairs = list(zip(cleaned_questions, answers))
        qa_pairs.extend(cur_qa_pairs)
    return qa_pairs

In [ ]:
# qa_pairs = generate_qa_pairs(
#     nodes[:1],
#     # nodes,
#     llm,
#     num_questions_per_chunk=10,
# )

In [ ]:
# qa_pairs

In [ ]:
# save
# import pickle

# pickle.dump(qa_pairs, open("eval_dataset.pkl", "wb"))

**Instead of regenerating the questions and answers pairs, runing the eval_dataset.pkl fie directly which was saved locally**

In [ ]:
# save
import pickle

qa_pairs = pickle.load(open("eval_dataset.pkl", "rb"))

In [ ]:
qa_pairs

## Evaluating Generation

### Correctness Evaluator

With openAI and qwen2-14B I tried to run the correctness evaluator on CORRECTNESS_SYS_TMPL but the output doesn't respect the prompt structure (confusing the reasoning with the score) then after changing the prompt it gives the prompt itself as an output!

**Alternative solutions:**

* Changing the prompt

* Changing the model (using llama-3.1-8b-instant   , DEEPSEEK R1 14B or GPT 4-o-mini )

https://www.deepseek.com/

In [ ]:
CORRECTNESS_SYS_TMPL = """
You are an expert evaluation system for a question answering chatbot.

You are given the following information:
a user query, a reference answer, and a generated answer.

Your job is to judge the relevance and correctness of the generated answer.

Give the score in the first line (just a number between 1 and 5) and the reasoning in another line please.

Please don't give the generated_answer in the output.

Follow these guidelines for scoring:
- Your score has to be between 1 and 5, where 1 is the worst and 5 is the best.
- If the generated answer is not relevant to the user query, \
you should give a score of 1.
- If the generated answer is relevant but contains mistakes, \
you should give a score between 2 and 3.
- If the generated answer is relevant and fully correct, \
you should give a score between 4 and 5.
"""

CORRECTNESS_USER_TMPL = """
{query}

{reference_answer}

{generated_answer}
"""

In [ ]:
eval_chat_template = ChatPromptTemplate(
    message_templates=[
        ChatMessage(role=MessageRole.SYSTEM, content=CORRECTNESS_SYS_TMPL),
        ChatMessage(role=MessageRole.USER, content=CORRECTNESS_USER_TMPL),
    ]
)

In [ ]:
from typing import List, Dict


def run_correctness_eval(
    query_str: str,
    reference_answer: str,
    generated_answer: str,
    llm: llm,
    threshold: float = 4.0,
) -> Dict:
    """Run correctness eval."""
    fmt_messages = eval_chat_template.format_messages(
        llm=llm,
        query=query_str,
        reference_answer=reference_answer,
        generated_answer=generated_answer,
    )
    chat_response = llm.chat(fmt_messages)
    raw_output = chat_response.message.content

    # Extract from response
    score_str, reasoning_str = raw_output.split("\n", 1)
    score = float(score_str)
    reasoning = reasoning_str.lstrip("\n")

    return {"passing": score >= threshold, "score": score, "reason": reasoning}

In [ ]:
query_str = "What is early determination of distribution policy?"
reference_answer = (
    "There are basically three options concerning distribution of software developed by the JRC or contributions made by the JRC to existing software:"
    "1) No distribution (software development for internal use only); "
    "2) « Proprietary » distribution (grant of individual licenses, either free of charge or with a license fee);"
    "3) distribution under an Open Source license."
    "There is no preferred “default”’ option as the policy reasons to choose one or another may vary from project to project."
    "Nevertheless, the decision must be taken in the  beginning of each project."
)

In [ ]:
# response = query_engine.query(query_str)
# generated_answer = str(response)

In [ ]:
# print(str(generated_answer))

In [ ]:
# import pickle

# pickle.dump(response, open("response.pkl", "wb"))

In [ ]:
response = pickle.load(open("response.pkl", "rb"))
generated_answer = str(response)
print(str(generated_answer))

In [ ]:
!pip install llama-index-llms-deepseek

**DeepSeek**

In [ ]:
from llama_index.llms.deepseek import DeepSeek


**Llama**

In [ ]:
!pip install llama-index-llms-huggingface
!pip install llama-index-embeddings-huggingface
!pip install llama-index-embeddings-huggingface-api

In [ ]:
#https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

In [ ]:
!huggingface-cli download meta-llama/Meta-Llama-3-8B-Instruct --token ""

In [ ]:
hf_token = ""
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    token=hf_token,
)

stopping_ids = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>"),
]

import torch
from llama_index.llms.huggingface import HuggingFaceLLM

llm2 = HuggingFaceLLM(
    model_name="meta-llama/Meta-Llama-3-8B-Instruct", 
    model_kwargs={
        "token": hf_token,
        "torch_dtype": torch.bfloat16,  # comment this line and uncomment below to use 4bit
        # "quantization_config": quantization_config
    },
    generate_kwargs={
        "do_sample": True,
        "temperature": 0.6,
        "top_p": 0.9,
    },
    tokenizer_name="meta-llama/Meta-Llama-3-8B-Instruct",
    tokenizer_kwargs={"token": hf_token},
    stopping_ids=stopping_ids,
)

In [ ]:
eval_results = run_correctness_eval(
    query_str, reference_answer, generated_answer, llm=llm2
)
display(eval_results)

### Faithfulness Evaluator

In [ ]:
EVAL_TEMPLATE = PromptTemplate(
    "Please tell if a given piece of information "
    "is supported by the context.\n"
    "You need to answer with either YES or NO.\n"
    "Answer YES if any of the context supports the information, even "
    "if most of the context is unrelated. "
    "Some examples are provided below. \n\n"
    "Information: Apple pie is generally double-crusted.\n"
    "Context: An apple pie is a fruit pie in which the principal filling "
    "ingredient is apples. \n"
    "Apple pie is often served with whipped cream, ice cream "
    "('apple pie à la mode'), custard or cheddar cheese.\n"
    "It is generally double-crusted, with pastry both above "
    "and below the filling; the upper crust may be solid or "
    "latticed (woven of crosswise strips).\n"
    "Answer: YES\n"
    "Information: Apple pies tastes bad.\n"
    "Context: An apple pie is a fruit pie in which the principal filling "
    "ingredient is apples. \n"
    "Apple pie is often served with whipped cream, ice cream "
    "('apple pie à la mode'), custard or cheddar cheese.\n"
    "It is generally double-crusted, with pastry both above "
    "and below the filling; the upper crust may be solid or "
    "latticed (woven of crosswise strips).\n"
    "Answer: NO\n"
    "Information: {query_str}\n"
    "Context: {context_str}\n"
    "Answer: "
)

EVAL_REFINE_TEMPLATE = PromptTemplate(
    "We want to understand if the following information is present "
    "in the context information: {query_str}\n"
    "We have provided an existing YES/NO answer: {existing_answer}\n"
    "We have the opportunity to refine the existing answer "
    "(only if needed) with some more context below.\n"
    "------------\n"
    "{context_msg}\n"
    "------------\n"
    "If the existing answer was already YES, still answer YES. "
    "If the information is present in the new context, answer YES. "
    "Otherwise answer NO.\n"
)

In [ ]:
from llama_index.core.response_synthesizers import Refine
from typing import List, Dict


def run_faithfulness_eval(
    generated_answer: str,
    contexts: List[str],
    llm: llm,
) -> Dict:
    """Run faithfulness eval."""

    refine = Refine(
        llm=llm,
        text_qa_template=EVAL_TEMPLATE,
        refine_template=EVAL_REFINE_TEMPLATE,
    )

    response_obj = refine.get_response(generated_answer, contexts)
    response_txt = str(response_obj)

    if "yes" in response_txt.lower():
        passing = True
    else:
        passing = False

    return {"passing": passing, "reason": str(response_txt)}

In [ ]:
response.source_nodes

In [ ]:
context_list = [n.get_content() for n in response.source_nodes]
eval_results = run_faithfulness_eval(
    generated_answer,
    contexts=context_list,
    llm=llm2,
)
display(eval_results)